# 05 – RQ3: restoring blurred and noisy drone footage
In this notebook, we degrade aerial frames with known blur and noise and then
restore them. We compare frequency-domain methods (the inverse filter,
the Wiener filter and Richardson-Lucy deconvolution) against spatial
filters. We measure what restoration recovers: pixel quality, but also detection
accuracy.

- Part A: we compute PSNR, SSIM and runtime for every method.
- Part B: we sweep the K parameter of the Wiener filter.
- Part C: we measure detection AP50 on clean, degraded and restored frames
  over the full VisDrone validation split.
- Part D (optional): we test on real blur from the GoPro dataset.


In [ ]:
# Install the packages we need. This notebook runs the detector and the
# restoration code only, so ultralytics is the single dependency.
!pip -q install ultralytics
import torch
print('CUDA available:', torch.cuda.is_available())

# Load the project code.
from pathlib import Path
SRC_ZIP = None
if SRC_ZIP is None:
    from google.colab import files
    up = files.upload()
    SRC_ZIP = next(iter(up))
!mkdir -p /content/project && unzip -q -o "$SRC_ZIP" -d /content/project
import sys
sys.path.insert(0, '/content/project/src')
sys.path.insert(0, '/content/project')

# Results are saved to Google Drive so they survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/sar_project_results'); OUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Download the VisDrone validation set and convert it to the person-only format.
%cd /content
!curl -sL -o vd_val.zip https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip
!mkdir -p data/raw && unzip -q -o vd_val.zip -d data/raw && rm vd_val.zip
import config
from pathlib import Path
config.RAW_DIR = Path('/content/data/raw'); config.DATA_DIR = Path('/content/data')
config.VISDRONE_VAL = config.RAW_DIR / 'VisDrone2019-DET-val'
config.VISDRONE_PERSON = config.DATA_DIR / 'visdrone_person'
from data import visdrone
visdrone.convert_split(config.VISDRONE_VAL, 'val', config.VISDRONE_PERSON)


In [ ]:
# Part A: measure PSNR, SSIM and runtime over 200 frames.
import cv2, eval_restore, eval_detect
eval_restore.VISDRONE_PERSON = config.VISDRONE_PERSON
eval_detect.VISDRONE_PERSON = config.VISDRONE_PERSON
imgs = []
for p in sorted((config.VISDRONE_PERSON/'images'/'val').glob('*.jpg'))[:200]:
    im = cv2.imread(str(p))
    if im is None: continue
    s = min(1.0, 1280/im.shape[1])
    imgs.append(cv2.resize(im, None, fx=s, fy=s) if s < 1 else im)
rows = eval_restore.run_intrinsic(imgs, out_csv=OUT/'restoration_intrinsic_full.csv')


In [ ]:
# Part B: run the Wiener K sweep, averaged over 20 frames.
import numpy as np, matplotlib.pyplot as plt
from restore import wiener_filter
from eval_restore import conditions, degrade_by, psnr
cond = next(c for c in conditions() if c['name']=='motion25_n5')
Ks = np.logspace(-4, 0, 13)
curves = []
for img in imgs[:20]:
    deg = degrade_by(cond, img, seed=0)
    curves.append([psnr(wiener_filter(deg, cond['psf'], K), img) for K in Ks])
mean = np.mean(curves, 0)
plt.figure(figsize=(6,4)); plt.semilogx(Ks, mean, marker='o')
plt.xlabel('Wiener K'); plt.ylabel('PSNR (dB)'); plt.grid(alpha=.3)
plt.title('Wiener K sensitivity (20 frames, motion25_n5)')
plt.savefig(OUT/'restoration_wiener_k_full.png', dpi=150)
print('Best K:', float(Ks[int(np.argmax(mean))]))


In [ ]:
# Part C: measure detection AP50 on clean, degraded and restored frames.
# The fine-tuned detector from notebook 01 must be in Drive for the AP50 to
# match the report; otherwise the run falls back to the COCO model and warns.
det_w = OUT / 'yolo11s_visdrone_best.pt'
if det_w.exists():
    det_w = str(det_w)
    print('Using fine-tuned detector:', det_w)
else:
    det_w = 'yolo11s.pt'
    print('WARNING: fine-tuned checkpoint not found at',
          OUT / 'yolo11s_visdrone_best.pt')
    print('Falling back to COCO-pretrained yolo11s.pt; the AP50 will NOT match')
    print('the report (clean AP50 0.709). Upload yolo11s_visdrone_best.pt to')
    print('that Drive folder and re-run this cell for the canonical numbers.')
from eval_restore import conditions, detection_under_degradation, write_csv
conds = {c['name']: c for c in conditions()}
rows  = detection_under_degradation(det_w, conds['motion25_n5'],
        ['wiener','rl20','unsharp'], max_images=548, device=0)
rows += detection_under_degradation(det_w, conds['noise25'],
        ['median','nlm','butterworth'], max_images=548, device=0)
write_csv(rows, OUT/'restoration_extrinsic_detection_full.csv')

In [ ]:
# Part D (optional): test on real blur from the GoPro dataset (about 9.5 GB).
# The real point spread functions are unknown, so only methods that do not
# need one apply fairly. The gap to the synthetic results motivates blind
# deconvolution as future work. This is off by default so 'Run all' is safe.
RUN_PART_D = False  # set True to download GoPro and run the real-blur test
if not RUN_PART_D:
    print('Part D skipped (set RUN_PART_D = True to run the 9.5 GB GoPro test).')
else:
    !pip -q install gdown && gdown 1H0PIXvJH4c40pk7ou6nAwoxuR4Qh_Sa2 -O /content/gopro.zip || echo 'download failed - skip Part D'
    import zipfile, glob, cv2, numpy as np
    from eval_restore import psnr, ssim
    from restore import unsharp_mask
    try:
        zipfile.ZipFile('/content/gopro.zip').extractall('/content/gopro')
        pairs = list(zip(sorted(glob.glob('/content/gopro/**/blur/**/*.png', recursive=True)),
                         sorted(glob.glob('/content/gopro/**/sharp/**/*.png', recursive=True))))[:100]
        rows = []
        for bp, sp in pairs:
            b, s = cv2.imread(bp), cv2.imread(sp)
            if b is None or s is None or b.shape != s.shape: continue
            u = unsharp_mask(b)
            rows.append((psnr(b,s), psnr(u,s), ssim(b,s), ssim(u,s)))
        if rows:
            m = np.mean(rows, 0)
            print(f'GoPro (n = {len(rows)}): PSNR blurred {m[0]:.2f} to unsharp {m[1]:.2f}; '
                  f'SSIM {m[2]:.3f} to {m[3]:.3f}')
    except Exception as e:
        print('Part D skipped:', e)

Together, the three parts answer the RQ3 question: whether restoration
recovers detection accuracy, or only pixel quality.
